<a href="https://colab.research.google.com/github/aiyman14/DACSS-758-Text-as-Data/blob/main/Final_project_check-in_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project Check In #2
## DACSS 758: Text as Data
## Aiyman Akbar

## Check In #1 Summary

The corpus that I am using for this project consists of news articles related to Bitcoin and cryptocurrency. I got this dataset from Hugging Face and it contains approximately 2,500 news articles. The articles are collected from various online news sources, including financial news websites, cryptocurrency publications, and general news outlets. Each entry in the dataset is a collection of different things from the article itself, such as the article title, summary text and author. These articles appear to have been aggregated from online news reporting about Bitcoin and cryptocurrency markets.

This dataset was probably created to support research in NLP, financial texts and analyzing them and related fields of research. Because it is an aggregation of crypto related articles it is easy to understand the relationship between digital assets and the media and how news narratives may influence market perception. However, the dataset may contain several sources of bias. Because news coverage tends to focus on major events such as market crashes or technological breakthroughs, this may lead to an overrepresentation of dramatic or high-impact stories. Additionally, different news outlets frame cryptocurrency differently depending on their audience or editorial stance.

I expect the language in the corpus to be pretty formal, in a journalistic style about financial and tech news. These types of articles generally do include jargon and specific terminology related to cryptocurrency, the markets, prices etc. I find this corpus interesting because of the relationship that it allows you to see between cryptocurrency and the media, and specifically how news narratives can influence the price of crypto and vice versa how the price fluctuations of crypto change the public sentiment or news headlines related to cryptocurrency. The corpus is structured as a standard dataset with each row being a single news article with many fields as the columns, and the total word count is around 230,813 words.

## Setup and Loading Data

In [ ]:
# install required libraries
!pip install datasets gensim wordcloud

In [ ]:
# imports
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud

# for preprocessing
from gensim.parsing.preprocessing import preprocess_string, strip_tags, strip_punctuation, strip_short
from gensim.parsing.preprocessing import strip_multiple_whitespaces, strip_numeric, remove_stopwords

In [ ]:
# load dataset directly from Hugging Face
ds = load_dataset("khaihernlow/bitcoin-news-articles-text-corpora")
df = pd.DataFrame(ds["train"])

In [ ]:
# explore the dataset
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head()

In [ ]:
# check for null values in key text columns
print("Null values in summary:", df['summary'].isnull().sum())
print("Null values in title:", df['title'].isnull().sum())
print("\nTopic distribution:")
print(df['topic'].value_counts())

In [ ]:
# drop rows with missing summaries since that is our main text field
df = df.dropna(subset=['summary'])
print("Shape after dropping nulls:", df.shape)

## Preprocessing

For this dataset, I am using the article "summary" field as my main text for analysis since it contains the most substantial text content for each article. The unit of analysis is a single article, represented by its summary. Below I will walk through each preprocessing step I chose to complete and explain why.

First, here is an example of what one article summary looks like before any preprocessing:

In [ ]:
# before preprocessing
summaries = df['summary'].tolist()
example_idx = 0

print("=" * 60)
print("BEFORE PREPROCESSING")
print("=" * 60)
print(summaries[example_idx])

### Step 1: Lowercasing

I am converting all text to lowercase so that words like "Bitcoin" and "bitcoin" are treated as the same token. Without this step the same word would be counted separately just because of capitalization which would mess up frequency counts and general analysis.

In [ ]:
# Step 1: lowercase
summaries_lower = [item.lower() for item in summaries]

### Step 2: Remove Punctuation

Punctuation does not really add any meaning for the kind of text analysis that I want to do with this dataset. Things like commas, periods, quotation marks etc would just add noise to the data and mess up tokenization, so I am remoivng all of them. 

### Step 3: Remove Extra Whitespace

After removing punctuation and other characters there will probably be extra spaces left over in certain places. Stripping multiple whitespaces should clean things up and makes sure tokens are separated properly.

### Step 4: Remove Stopwords

Stopwords are common words like "the", "is", "and", "of" etc. that appear very frequently but will not carry much meaning for my analysis. If I kept them in they would dominate the word frequencies and make it harder to see the actual important terms related to Bitcoin and cryptocurrency news. Removing them will let the more meaningful content words stand out.

### Step 5: Remove Short Words

I am also removing words that are shorter than 3 characters. These could be leftover fragments or very common words that wont really add value to my analysis, like the stopwords from step 4. This will be like a second layer after removing the stopwords. 

In [ ]:
# Steps 2-5: 
CUSTOM_FILTERS = [strip_punctuation, strip_multiple_whitespaces, remove_stopwords, strip_short]
summaries_toks = [preprocess_string(item, CUSTOM_FILTERS) for item in summaries_lower]
print(summaries_toks[0])

### Deliberately Omitted Steps

I chose not to remove numbers because in financial news articles numbers like prices and percentages can be meaningful, but I may reconsider this depending on what I finaly decide to look for in my analysis later in the process. 

In [ ]:
# merge tokens back after steps 2-5 into strings for the upcoming steps
summaries_toks_merged = [' '.join(item) for item in summaries_toks]

## After Preprocessing Example

Now here is the same article after all the preprocessing steps have been applied:

In [ ]:
# same article AFTER preprocessing
print("=" * 60)
print("AFTER PREPROCESSING")
print("=" * 60)
print(summaries_toks_merged[example_idx])

## Visualizations

### Visualization 1: Top 25 Most Frequent Words

This bar chart shows the 25 most common words across all the article summaries after preprocessing. This is helpful because it gives a clear picture of what the dominant themes and topics are in Bitcoin news coverage. If certain words show up way more than others, that says something about what the media focuses on when they talk about Bitcoin and cryptocurrency.

In [ ]:
# all tokens into one list
all_tokens = [item for sublist in summaries_toks for item in sublist]
print("Total tokens:", len(all_tokens))

# count frequencies
c = Counter(all_tokens)
word_df = pd.DataFrame(c.items(), columns=['Token', 'Count'])
word_df = word_df.sort_values(by='Count', ascending=False)
word_df.head(10)

In [ ]:
# plot top 25 most frequent words
top25 = word_df.head(25)

plt.figure(figsize=(12, 6))
plt.bar(top25['Token'], top25['Count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Token')
plt.ylabel('Frequency')
plt.title('Top 25 Most Frequent Words in Bitcoin News Summaries')
plt.tight_layout()
plt.show()

### Visualization 2: Word Cloud

I like the world cloud as a way to visualize the same frequency data but in a more visually interesting way. This format makes it easy to see at a glance what the most prominent terms are in the dataset. It is also just a nice visual to include in my opinion because it immediately communicates what the corpus is about.

In [ ]:
# create word cloud from all preprocessed text
all_text = ' '.join(summaries_toks_merged)

wordcloud = WordCloud(width=800, height=400,
                      background_color='white',
                      colormap='viridis',
                      max_words=100).generate(all_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Bitcoin News Article Summaries')
plt.tight_layout()
plt.show()

## Next Steps

Moving forward with this data, I would like to explore sentiment analysis of the Bitcoin news articles. One potential approach would be to use a sentiment dictionary like VADER or AFINN to score each article summary and see the overall distribution of positive, negative and neutral sentiment across the corpus. This would help answer the question of whether Bitcoin news coverage tends to lean more positive or negative overall.

Another thing I am interested in is looking at how the language and sentiment might differ across the different topic categories in the dataset, like "news" vs "finance" vs "tech". Since different types of outlets and topics might frame Bitcoin differently, comparing the sentiment and word usage across these categories could reveal some interesting patterns about how Bitcoin is discussed in different contexts.

## Challenges

One challenge I encountered while working with this data is the potential for domain-specific terms to affect the analysis. Words like "bitcoin", "crypto", and "blockchain" are going to dominate the frequency counts because that is literally what every article is about, so it might be worth considering whether to filter those out in future analysis to see what other themes emerge beyond the obvious ones. 

Going forward, I anticipate that choosing the right approach for sentiment analysis might be tricky. General purpose sentiment dictionaries might not capture the nuances of financial and crypto-specific language very well. A word like "crash" has a very negative connotation in general but in crypto news it is used very commonly to describe price movements. This is something I have been thinking about and will need figure out what to do with as I move forward with this project.